# Bài 16: Dữ liệu cá nhân
**Mở hộp đen AI · Tập 2 · Phòng thí nghiệm Colab**

Mọi người, mọi con số trong notebook này đều **do máy tạo ngẫu nhiên**, không phải người thật. Luật Bảo vệ dữ liệu cá nhân (2025) cấm tái nhận dạng dữ liệu đã khử nhận dạng của người thật, trừ khi pháp luật có quy định khác: chỉ tập trên dữ liệu giả như ở đây.
Ebook Bên trong cỗ máy Trí tuệ nhân tạo - Tập 2. Phát hành tại https://montinhoc.vn

## Chặng 1: Chạy
### Ô 1. Một bảng khám sức khoẻ tưởng tượng của 300 học sinh

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(16)
n = 300
HO = ["Nguyễn", "Trần", "Lê", "Phạm", "Hoàng", "Vũ", "Đặng", "Bùi"]
TEN = {"Nam": ["Bình", "Dũng", "Huy", "Khoa", "Minh", "Phúc", "Quân", "Tú"],
       "Nữ": ["An", "Chi", "Hà", "Lan", "Linh", "Ngọc", "Thảo", "Vy"]}
PHUONG = ["Phường Đông", "Phường Tây", "Phường Nam", "Phường Bắc", "Xã Hoà"]
gioi = rng.choice(["Nam", "Nữ"], n)
so_ngay = rng.integers(0, 3 * 365, n)                  # sinh trong 3 năm
ngay_sinh = pd.Timestamp("2009-01-01") + pd.to_timedelta(so_ngay, unit="D")
bang = pd.DataFrame({
    "ho_ten": [f"{rng.choice(HO)} {rng.choice(TEN[g])} ({i})"
               for i, g in enumerate(gioi)],
    "ngay_sinh": ngay_sinh.strftime("%d/%m/%Y"),
    "gioi_tinh": gioi,
    "phuong": rng.choice(PHUONG, n),
    "hen_suyen": rng.choice(["Có", "Không"], n, p=[0.1, 0.9]),
})
print(bang.head(3))

### Ô 2. “Ẩn danh” kiểu ngây thơ: xoá cột họ tên

In [ ]:
an_danh = bang.drop(columns="ho_ten")
print(an_danh.head(3))
print("Số dòng:", len(an_danh))

### Ô 3. Một bảng công khai khác: danh sách sinh nhật của câu lạc bộ
Có họ tên, ngày sinh, giới tính, phường của 60 bạn. Không có thông tin sức khoẻ.

In [ ]:
clb = bang.sample(60, random_state=1)
clb = clb[["ho_ten", "ngay_sinh", "gioi_tinh", "phuong"]]
khoa = ["ngay_sinh", "gioi_tinh", "phuong"]
ghep = clb.merge(an_danh, on=khoa)          # nối theo 3 cột chung
dem = ghep.groupby("ho_ten").size()
nhan_ra = dem[dem == 1].index               # khớp đúng 1 dòng
print("Số bạn trong danh sách câu lạc bộ:", len(clb))
print("Số bạn bị nhận ra lại:", len(nhan_ra))
print(ghep[ghep.ho_ten.isin(nhan_ra) & (ghep.hen_suyen == "Có")].head(3))

## Chặng 2: Đọc
1. Ở ô 2, cột nào đã bị xoá? Cột nào vẫn còn?
2. Ở ô 3, hàm `merge` nối hai bảng bằng những cột nào?
3. Vì sao chỉ cần ba cột “vô hại” là đủ nhận ra lại một người?

## Chặng 3: Sửa
### Ô 4. Bao nhiêu người là “duy nhất” trong bảng?

In [ ]:
def ti_le_duy_nhat(df, cot):
    # mỗi dòng: nhóm của nó có mấy người
    co_nhom = df.groupby(cot)[cot[0]].transform("size")
    return (co_nhom == 1).mean()

an_danh["nam_sinh"] = an_danh.ngay_sinh.str[-4:]
for cot in [["gioi_tinh"], ["gioi_tinh", "phuong"],
            ["gioi_tinh", "phuong", "nam_sinh"],
            ["gioi_tinh", "phuong", "ngay_sinh"]]:
    ti_le = ti_le_duy_nhat(an_danh, cot)
    print(f"{' + '.join(cot):35}: {ti_le:6.1%} là duy nhất")

### Ô 5. Làm mờ cho đến khi mỗi nhóm có ít nhất k người

In [ ]:
mo = an_danh[["nam_sinh", "gioi_tinh", "phuong", "hen_suyen"]].copy()
k = mo.groupby(["nam_sinh", "gioi_tinh", "phuong"]).size().min()
print("Bỏ ngày tháng, giữ năm sinh -> nhóm nhỏ nhất có", k, "người")

clb["nam_sinh"] = clb.ngay_sinh.str[-4:]
ghep2 = clb.merge(mo, on=["nam_sinh", "gioi_tinh", "phuong"])
dem2 = ghep2.groupby("ho_ten").size()
print("Số bạn bị nhận ra lại:", (dem2 == 1).sum())
print("Số dòng khớp trung bình cho mỗi bạn:", round(dem2.mean(), 1))

### Ô 6. Cái giá của việc làm mờ

In [ ]:
print("Tỉ lệ hen suyễn theo phường (vẫn tính được):")
co_benh = mo.hen_suyen == "Có"
print(co_benh.groupby(mo.phuong).mean().round(2))
print("Tỉ lệ theo tháng sinh: không còn tính được, vì đã bỏ ngày tháng.")

### Ô 7. Không cần bảng nào bị lộ: đoán điều em chưa từng nói
500 học sinh tưởng tượng, mỗi bạn bấm “thích” một số trong 20 trang trên mạng xã hội. Máy đoán xem bạn ấy định bầu ai làm lớp trưởng.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

m, so_trang = 500, 20
ung_ho_A = rng.integers(0, 2, m)         # bí mật: 1 = định bầu bạn A
lech = rng.normal(0, 0.5, so_trang)      # mỗi trang hơi nghiêng một phía
p = 1 / (1 + np.exp(-(np.outer(2 * ung_ho_A - 1, lech) - 1)))
thich = (rng.random((m, so_trang)) < p).astype(int)    # 500 x 20

Ta, Tb, ya, yb = train_test_split(thich, ung_ho_A, test_size=0.3,
                                  random_state=0)
mo_hinh = LogisticRegression(max_iter=1000).fit(Ta, ya)
print(f"Đoán đúng người định bầu ai: {mo_hinh.score(Tb, yb):.1%}")
print(f"Đoán bừa (tung đồng xu):      {max(yb.mean(), 1 - yb.mean()):.1%}")

## Chặng 4: Tự làm
1. Ở ô 5, làm mờ thêm: gộp 5 phường thành 2 vùng (“nội thành”, “ngoại thành”). Nhóm nhỏ nhất có bao nhiêu người? Còn ai bị nhận ra không?
2. Ở ô 7, chỉ dùng 5 trang đầu (`thich[:, :5]`). Máy còn đoán đúng bao nhiêu?
3. Viết hàm `kiem_tra_k(df, cot)` in ra nhóm nhỏ nhất và những dòng thuộc nhóm đó.

**Nhật ký dùng AI:** Em có nhờ AI ở bước nào không? Nó sai ở đâu? Em sửa thế nào?

*(ghi vào đây)*